# 1. Rozeznanie i Setup

In [1]:
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import requests
from io import BytesIO
import random

plt.rcParams['figure.figsize'] = (15, 10)
%matplotlib inline

print("✓ Biblioteki załadowane")

/Users/martasolarz/phd/MapQualityEvaluator/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Biblioteki załadowane


In [17]:
def load_image_from_url(url, timeout=5):
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return Image.open(BytesIO(response.content))
    except Exception as e:
        return None

def check_url(url, timeout=5):
    try:
        response = requests.head(url, timeout=timeout, allow_redirects=True)
        if response.status_code == 200:
            return 'OK'
        else:
            return 'ERR'
    except:
        return 'ERR'

In [ ]:
dataset = load_dataset("sraimund/MapPool", split="train", streaming=True)

In [18]:
sample_100k = list(dataset.take(100000))

print(f"✓ Pobrano {len(sample_100k)} map")

print("\nLosowanie 10 000 map z pobranej próbki...")
random.seed(42)
sample_10k = random.sample(sample_100k, 10000)

print(f"✓ Wylosowano {len(sample_10k)} losowych map")

✓ Pobrano 100000 map

Losowanie 10 000 map z pobranej próbki...
✓ Wylosowano 10000 losowych map


In [19]:
data = []
for item in tqdm(sample_10k, desc="Przetwarzanie map"):
    url_status = check_url(item.get('url', ''))

    row = {
        'uid': item.get('uid'),
        'url': item.get('url'),
        'url_status': url_status,
        'text': item.get('text'),
        'original_width': item.get('original_width'),
        'original_height': item.get('original_height'),
        'clip_l14_similarity_score': item.get('clip_l14_similarity_score'),
        'sha256': item.get('sha256')
    }

    data.append(row)

df = pd.DataFrame(data)

print(f"\n✓ Utworzono DataFrame: {len(df)} wierszy, {len(df.columns)} kolumn")


Konwersja do DataFrame i sprawdzanie URL...
UWAGA: Sprawdzanie 10k URL może potrwać 10-20 minut!



Przetwarzanie map: 100%|██████████| 10000/10000 [2:41:58<00:00,  1.03it/s]  



✓ Utworzono DataFrame: 10000 wierszy, 8 kolumn


In [20]:
url_ok = (df['url_status'] == 'OK').sum()
url_err = (df['url_status'] == 'ERR').sum()

print("STATYSTYKI URL:")
print(f"Działające (OK):  {url_ok:5} ({url_ok/len(df)*100:.1f}%)")
print(f"Niedziałające (ERR): {url_err:5} ({url_err/len(df)*100:.1f}%)")

STATYSTYKI URL:
Działające (OK):   5475 (54.8%)
Niedziałające (ERR):  4525 (45.2%)


In [22]:
filename = 'sample_10k_maps_with_url_status.csv'
df.to_csv(filename, index=False, encoding='utf-8')

print(f"\n✓ Zapisano do: {filename}")


✓ Zapisano do: sample_10k_maps_with_url_status.csv


In [1]:
import pandas as pd

df = pd.read_csv('../data/sample_10k_maps_with_url_status.csv')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10002 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   uid                        10000 non-null  str    
 1   url                        10002 non-null  str    
 2   url_status                 10002 non-null  str    
 3   text                       9993 non-null   str    
 4   original_width             10000 non-null  str    
 5   original_height            9998 non-null   float64
 6   clip_l14_similarity_score  9998 non-null   float64
 7   sha256                     9998 non-null   str    
dtypes: float64(2), str(6)
memory usage: 3.2 MB


In [3]:
df_url_ok = df[df['url_status'] == 'OK'].reset_index(drop=True)

In [4]:
df_url_ok.head()

,uid,url,url_status,text,original_width,original_height,clip_l14_similarity_score,sha256
0,eae4819aef3d65027b8d28a230afc0f9,https://cdn1.vectorstock.com/i/thumbs/22/75/fr...,OK,French Southern Territories Rubber Stamps vect...,112,118.0,0.362793,2ca3b7f2d2f2996a3800c001797ed3c7ee666ca82dbb87...
1,4ea106e4a5867df0f987300f2038cd4c,http://7.hikb.at:80/img/iconmenu/attraction.png,OK,New York - Látnivalók,64,64.0,0.127808,c3f92275db5f9b35f2efe57d85fc0f547347b1fd361646...
2,030e8a94dd1934c8c20eb85cc04d009d,http://images.slideplayer.com/9/2524694/slides...,OK,2004 Weekday Arterial Hourly Volumes: 6 p.m. t...,960,720.0,0.451660,98e4271537a8193cef6ba4a5f3f64ba60f49d11aa19972...
3,f8b58c9d96f8ea0a0e49493da6417b5f,http://tse1.mm.bing.net/th?id=OIP.1ukV5UWF-YxK...,OK,house plans less than 2000 square feet in kera...,474,474.0,0.252686,404b4ec335b636fa655123e974230f2c121634b9550efb...
4,e6c1b51c0bcd4f5c45a8c38af79481ff,https://bt-photos.global.ssl.fastly.net/tulsa/...,OK,https://bt-photos.global.ssl.fastly.net/tulsa/...,640,360.0,0.236816,34d3a4fc92b1103fed18cc38c59b165ca3fc30f46d8665...


In [5]:
df_url_ok.info()

<class 'pandas.DataFrame'>
RangeIndex: 5475 entries, 0 to 5474
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   uid                        5475 non-null   str    
 1   url                        5475 non-null   str    
 2   url_status                 5475 non-null   str    
 3   text                       5473 non-null   str    
 4   original_width             5474 non-null   str    
 5   original_height            5474 non-null   float64
 6   clip_l14_similarity_score  5474 non-null   float64
 7   sha256                     5474 non-null   str    
dtypes: float64(2), str(6)
memory usage: 1.7 MB


In [16]:
import ipywidgets as widgets
from IPython.display import display, HTML, Image
import requests
from io import BytesIO
import numpy as np

class MapViewer:
    def __init__(self, df, shuffle=True):
        self.df = df.reset_index(drop=True)
        
        if shuffle:
            self.indices = np.random.permutation(len(self.df))
        else:
            self.indices = np.arange(len(self.df))
        
        self.current_position = 0
        
        self.image_widget = widgets.Output()
        self.info_widget = widgets.HTML()
        self.prev_button = widgets.Button(description='← Previous')
        self.next_button = widgets.Button(description='Next →')
        self.shuffle_button = widgets.Button(description='🔀 Shuffle')
        self.counter_widget = widgets.HTML()
        
        self.prev_button.on_click(self.show_previous)
        self.next_button.on_click(self.show_next)
        self.shuffle_button.on_click(self.reshuffle)
        
        nav_buttons = widgets.HBox([self.prev_button, self.counter_widget, self.next_button])
        buttons = widgets.VBox([nav_buttons, self.shuffle_button])
        self.container = widgets.VBox([buttons, self.info_widget, self.image_widget])
        
    def get_current_index(self):
        return self.indices[self.current_position]
    
    def show_image(self):
        self.image_widget.clear_output(wait=True)
        
        if self.current_position < 0 or self.current_position >= len(self.indices):
            return
        
        row = self.df.iloc[self.get_current_index()]
        
        self.counter_widget.value = f"<center><b>Mapa {self.current_position + 1} / {len(self.df)}</b></center>"
        
        info_html = f"""
        <div style='background-color: #f0f0f0; padding: 10px; border-radius: 5px; margin: 10px 0;'>
            <b>Text:</b> {row['text']}<br>
            <b>URL:</b> <a href="{row['url']}" target="_blank">{row['url']}</a><br>
            <b>Wymiary:</b> {row['original_width']} x {row['original_height']}<br>
            <b>UID:</b> {row['uid']}<br>
            <b>Oryginalny indeks:</b> {self.get_current_index()}
        </div>
        """
        self.info_widget.value = info_html
        
        with self.image_widget:
            try:
                display(HTML(f'<img src="{row["url"]}" style="max-width: 800px; border: 2px solid #ddd; border-radius: 5px;">'))
            except Exception as e:
                print(f"Błąd ładowania obrazka: {e}")
                print(f"URL: {row['url']}")
    
    def show_next(self, b):
        if self.current_position < len(self.indices) - 1:
            self.current_position += 1
            self.show_image()
    
    def show_previous(self, b):
        if self.current_position > 0:
            self.current_position -= 1
            self.show_image()
    
    def reshuffle(self, b):
        self.indices = np.random.permutation(len(self.df))
        self.current_position = 0
        self.show_image()
    
    def display(self):
        self.show_image()
        display(self.container)

viewer = MapViewer(df_url_ok, shuffle=True)
viewer.display()

In [8]:
import pandas as pd

# Lista słów kluczowych
keywords = [
    'choropleth',
    'county',
    'population',
    'density',
    'statistical',
    'demographic',
    'demographical',
    'distribution',
    'region',
    'municipality',
    'people',
    'area',
    'chart',
    'pie',
    'stats',
    'survey',
    'census',
    'bureau',
    'revenue',
    'global',
    'comparison',
    'thematic',
    'average',
    'age'
]

# Funkcja sprawdzająca czy text zawiera któreś ze słów kluczowych
def contains_keywords(text, keywords):
    if pd.isna(text):  # obsługa wartości NaN
        return False
    text_lower = str(text).lower()
    return any(keyword.lower() in text_lower for keyword in keywords)

# Filtrowanie
df_filtered = df_url_ok[df_url_ok['text'].apply(lambda x: contains_keywords(x, keywords))].copy()

# Statystyki
print(f"Oryginalny zbiór: {len(df_url_ok)} wierszy")
print(f"Przefiltrowany zbiór: {len(df_filtered)} wierszy")
print(f"Odfiltrowano: {len(df_url_ok) - len(df_filtered)} wierszy ({100 * (len(df_url_ok) - len(df_filtered)) / len(df_url_ok):.1f}%)")

# Wyświetl przykłady
print("\n--- Przykłady przefiltrowanych map ---")
print(df_filtered[['text', 'url_status']].head(10))

# Możesz też zobaczyć, które słowa kluczowe były najczęstsze
print("\n--- Statystyki słów kluczowych ---")
keyword_counts = {}
for keyword in keywords:
    count = df_filtered['text'].str.lower().str.contains(keyword, na=False).sum()
    if count > 0:
        keyword_counts[keyword] = count

for keyword, count in sorted(keyword_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{keyword}: {count}")

Oryginalny zbiór: 5475 wierszy
Przefiltrowany zbiór: 639 wierszy
Odfiltrowano: 4836 wierszy (88.3%)

--- Przykłady przefiltrowanych map ---
                                                  text url_status
0    French Southern Territories Rubber Stamps vect...         OK
3    house plans less than 2000 square feet in kera...         OK
4    https://bt-photos.global.ssl.fastly.net/tulsa/...         OK
7                                     Screenshot Image         OK
38          Denver Metro - Services Area for Dumpsters         OK
48   Ward County Texas Map Texas County Highway Map...         OK
85   Silent Bloc de biellettes de commande embrayag...         OK
92                dating app global comparison revenue         OK
100  silhouette country borders map of united kingd...         OK
108  Map v -Second Half - Distribution of Cities in...         OK

--- Statystyki słów kluczowych ---
age: 449
global: 48
county: 39
area: 32
pie: 26
chart: 21
region: 20
population: 12
people: 12
dis

In [9]:
import pandas as pd
import re

# Pozytywne słowa kluczowe (muszą być obecne)
positive_keywords = [
    'choropleth',
    'county',
    'population',
    'density',
    'statistical',
    'demographic',
    'demographical',
    'distribution',
    'region',
    'municipality',
    'people',
    'area',
    'chart',
    'pie',
    'stats',
    'survey',
    'census',
    'bureau',
    'revenue',
    'global',
    'comparison',
    'thematic',
    'average',
    'age'
]

# Negatywne słowa kluczowe (NIE mogą być obecne)
negative_keywords = [
    'road',
    'route',
    'street',
    'photo',
    'photos',
    'hiking',
    'satellite',
    'plan',
    'plans',
    'image',
    'images',
    'stamps',
    'stamp',
    'things to do',
    'building',
    'buildings',
    'screenshot',
    'fotos',
    'foto',
    'illustratie',
    'old map',
    'metro',
    'floor plan',
    'floor plans',
    'planos',
    'surface',
    'photography',
    'city',
    'cities',
    'cloud cover',
    '3d',
    'wallpaper',
    'aerial view',
    'Encyclopedia Britannica',
    'Ground Floor',
    'floorplan',
    'illustration',
    'thumbnail',
    'vectorpictogram',
    'icon',
    'icons',
    'clicking',
    'Апартамент',
    'labeled continents'
]

# Stwórz regex patterns
positive_pattern = '|'.join([re.escape(keyword) for keyword in positive_keywords])
negative_pattern = '|'.join([re.escape(keyword) for keyword in negative_keywords])

# Filtruj: zawiera pozytywne słowa
df_with_positive = df_url_ok[
    df_url_ok['text'].str.contains(positive_pattern, case=False, na=False, regex=True)
].copy()

# Filtruj: NIE zawiera negatywnych słów
df_filtered = df_with_positive[
    ~df_with_positive['text'].str.contains(negative_pattern, case=False, na=False, regex=True)
].copy()

# Statystyki
print(f"Oryginalny zbiór: {len(df_url_ok)} wierszy")
print(f"Po filtrze pozytywnym: {len(df_with_positive)} wierszy")
print(f"Po filtrze negatywnym: {len(df_filtered)} wierszy")
print(f"Pozostało: {100 * len(df_filtered) / len(df_url_ok):.1f}%")
print(f"Odfiltrowano negatywnych: {len(df_with_positive) - len(df_filtered)} wierszy")

# Przykłady
print("\n--- Przykłady przefiltrowanych map ---")
print(df_filtered[['text', 'url_status']].head(10))

# Statystyki słów kluczowych
print("\n--- Najczęstsze pozytywne słowa ---")
positive_counts = {}
for keyword in positive_keywords:
    count = df_filtered['text'].str.lower().str.contains(re.escape(keyword), na=False).sum()
    if count > 0:
        positive_counts[keyword] = count

for keyword, count in sorted(positive_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{keyword}: {count}")

# Sprawdź które negatywne słowa były najczęściej odfiltrowane
print("\n--- Najczęściej odfiltrowane negatywne słowa ---")
negative_counts = {}
for keyword in negative_keywords:
    count = df_with_positive['text'].str.lower().str.contains(re.escape(keyword), na=False).sum()
    if count > 0:
        negative_counts[keyword] = count

for keyword, count in sorted(negative_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{keyword}: {count}")

Oryginalny zbiór: 5475 wierszy
Po filtrze pozytywnym: 639 wierszy
Po filtrze negatywnym: 234 wierszy
Pozostało: 4.3%
Odfiltrowano negatywnych: 405 wierszy

--- Przykłady przefiltrowanych map ---
                                                  text url_status
48   Ward County Texas Map Texas County Highway Map...         OK
85   Silent Bloc de biellettes de commande embrayag...         OK
92                dating app global comparison revenue         OK
141  week 7 tutorial comm101 Comm101 - principles o...         OK
186                 Locations of visitors to this page         OK
199                Excerpt from map of Chenango County         OK
201  Fig. 14. GPR survey in the podium. Time-slices...         OK
203  global-warming,-jet-stream,-vortice-polare,-fe...         OK
228                 Ukraine_census_2001_Ukrainians.svg         OK
237  Voyage 224 L 238 Le De Minorque Bal 233 Ares E...         OK

--- Najczęstsze pozytywne słowa ---
age: 105
county: 31
area: 21
chart: 19
pie

In [14]:

!pip install langdetect

from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # Dla powtarzalnych wyników

def detect_language(text):
    """Wykrywa język tekstu"""
    if pd.isna(text) or str(text).strip() == '':
        return 'unknown'
    try:
        return detect(str(text))
    except:
        return 'unknown'

df_url_ok['detected_lang'] = df_url_ok['text'].apply(detect_language)

# Zobacz rozkład języków
print("Rozkład języków:")
print(df_url_ok['detected_lang'].value_counts())

# Filtruj używając odpowiednich słów kluczowych dla każdego języka
def matches_criteria(row):
    """Sprawdza czy wiersz spełnia kryteria w odpowiednim języku"""
    text = str(row['text']).lower() if not pd.isna(row['text']) else ''
    lang = row['detected_lang']
    
    # Wybierz odpowiednie słowa kluczowe
    pos_keywords = positive_keywords.get(lang, positive_keywords['en'])
    neg_keywords = negative_keywords.get(lang, negative_keywords['en'])
    
    # Sprawdź pozytywne
    has_positive = any(kw.lower() in text for kw in pos_keywords)
    
    # Sprawdź negatywne
    has_negative = any(kw.lower() in text for kw in neg_keywords)
    
    return has_positive and not has_negative

df_filtered = df_url_ok[df_url_ok.apply(matches_criteria, axis=1)].copy()

print(f"\nZnaleziono {len(df_filtered)} map spełniających kryteria")
print("\nRozkład języków w przefiltrowanym zbiorze:")
print(df_filtered['detected_lang'].value_counts())

  Using cached langdetect-1.0.9.tar.gz (981 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993335 sha256=aa3682befbf6bcdcc847e6d06436bea2e48f46944bdf88240b328cb80b39b985
  Stored in directory: /Users/martasolarz/Library/Caches/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Rozkład języków:
detected_lang
en         2390
fr          280
de          257
ru          190
nl          169
it          169
tl          153
es          148
ja          143
unknown     134
ko          123
zh-cn        97
pt           95
id           89
ro           89
ca           71
pl           66
sw           64
no           59
af           57
bg           56
sl           55
da    

AttributeError: 'list' object has no attribute 'get'

In [15]:
import pandas as pd
import re
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

# Główne języki z tłumaczeniami (pokrywają ~85% danych)
positive_keywords = {
    'en': ['choropleth', 'county', 'population', 'density', 'statistical', 
           'demographic', 'distribution', 'region', 'municipality', 'people',
           'area', 'chart', 'pie', 'stats', 'survey', 'census', 'bureau',
           'revenue', 'global', 'comparison', 'thematic', 'average', 'age'],
    
    'fr': ['population', 'densité', 'statistique', 'démographique',
           'distribution', 'région', 'municipalité', 'gens', 'zone',
           'graphique', 'enquête', 'recensement', 'comparaison',
           'moyenne', 'âge', 'thématique', 'carte choroplèthe'],
    
    'de': ['bevölkerung', 'dichte', 'statistisch', 'demografisch', 
           'verteilung', 'region', 'gemeinde', 'menschen', 'gebiet',
           'diagramm', 'umfrage', 'zensus', 'vergleich', 'durchschnitt',
           'alter', 'thematisch', 'landkreis'],
    
    'ru': ['население', 'плотность', 'статистический', 'демографический',
           'распределение', 'регион', 'муниципалитет', 'люди', 'область',
           'график', 'опрос', 'перепись', 'сравнение', 'средний',
           'возраст', 'тематический', 'округ', 'район'],
    
    'nl': ['bevolking', 'dichtheid', 'statistisch', 'demografisch',
           'verdeling', 'regio', 'gemeente', 'mensen', 'gebied',
           'grafiek', 'enquête', 'volkstelling', 'vergelijking',
           'gemiddelde', 'leeftijd', 'thematisch'],
    
    'it': ['popolazione', 'densità', 'statistico', 'demografico',
           'distribuzione', 'regione', 'municipio', 'persone', 'area',
           'grafico', 'sondaggio', 'censimento', 'confronto', 'media',
           'età', 'tematico', 'provincia'],
    
    'es': ['población', 'densidad', 'estadístico', 'demográfico',
           'distribución', 'región', 'municipio', 'gente', 'área',
           'gráfico', 'encuesta', 'censo', 'comparación', 'promedio',
           'edad', 'temático', 'provincia'],
    
    'pl': ['populacja', 'ludność', 'gęstość', 'statystyczny', 'statystyka',
           'demograficzny', 'rozkład', 'dystrybucja', 'region', 'gmina',
           'ludzie', 'obszar', 'wykres', 'badanie', 'spis', 'przeciętny',
           'średni', 'wiek', 'porównanie', 'tematyczny', 'powiat', 'województwo'],
    
    'pt': ['população', 'densidade', 'estatístico', 'demográfico',
           'distribuição', 'região', 'município', 'pessoas', 'área',
           'gráfico', 'pesquisa', 'censo', 'comparação', 'média',
           'idade', 'temático'],
    
    'ja': ['人口', '密度', '統計', '人口統計', '分布', '地域', '市町村',
           '人々', 'エリア', 'グラフ', '調査', '国勢調査', '比較', '平均', '年齢'],
    
    'zh-cn': ['人口', '密度', '统计', '人口统计', '分布', '地区', '市镇',
              '人民', '区域', '图表', '调查', '人口普查', '比较', '平均', '年龄'],
    
    'ko': ['인구', '밀도', '통계', '인구통계', '분포', '지역', '시군',
           '사람들', '면적', '차트', '조사', '인구조사', '비교', '평균', '나이'],
}

negative_keywords = {
    'en': ['road', 'route', 'street', 'photo', 'photos', 'hiking', 'satellite',
           'plan', 'plans', 'image', 'images', 'stamp', 'stamps', 'building',
           'buildings', 'screenshot', 'old map', 'metro', 'floor plan',
           'surface', 'photography', 'city', 'cities', 'cloud cover', '3d',
           'wallpaper', 'aerial view', 'illustration', 'thumbnail', 'icon',
           'icons', 'labeled continents', 'things to do'],
    
    'fr': ['route', 'rue', 'photo', 'photos', 'satellite', 'plan', 'plans',
           'image', 'images', 'timbre', 'bâtiment', 'bâtiments', 'métro',
           'photographie', 'ville', 'villes', 'icône', 'vignette', 'illustration',
           'vue aérienne', 'ancienne carte'],
    
    'de': ['straße', 'foto', 'fotos', 'satellit', 'plan', 'pläne',
           'bild', 'bilder', 'stempel', 'gebäude', 'metro', 'grundriss',
           'fotografie', 'stadt', 'städte', 'symbol', 'miniatur', 'illustration',
           'luftaufnahme', 'alte karte'],
    
    'ru': ['дорога', 'улица', 'фото', 'спутник', 'план', 'планы',
           'изображение', 'марка', 'здание', 'метро', 'фотография',
           'город', 'города', 'иконка', 'миниатюра', 'иллюстрация',
           'старая карта'],
    
    'nl': ['weg', 'straat', 'foto', "foto's", 'satelliet', 'plan', 'plannen',
           'afbeelding', 'afbeeldingen', 'postzegel', 'gebouw', 'gebouwen',
           'metro', 'fotografie', 'stad', 'steden', 'pictogram', 'illustratie',
           'oude kaart'],
    
    'it': ['strada', 'via', 'foto', 'satellite', 'piano', 'piani',
           'immagine', 'immagini', 'francobollo', 'edificio', 'edifici',
           'metro', 'planimetria', 'fotografia', 'città', 'icona', 'miniatura',
           'illustrazione', 'mappa antica'],
    
    'es': ['carretera', 'calle', 'foto', 'fotos', 'satélite', 'plan', 'planos',
           'imagen', 'imágenes', 'sello', 'edificio', 'edificios', 'metro',
           'plano', 'fotografía', 'ciudad', 'ciudades', 'icono', 'miniatura',
           'ilustración', 'mapa antiguo'],
    
    'pl': ['droga', 'ulica', 'zdjęcie', 'zdjęcia', 'foto', 'satelita',
           'plan', 'plany', 'obraz', 'pieczęć', 'budynek', 'budynki',
           'metro', 'plan piętra', 'fotografia', 'miasto', 'miasta',
           'ikona', 'ikony', 'miniatura', 'ilustracja', 'stara mapa'],
    
    'pt': ['estrada', 'rua', 'foto', 'fotos', 'satélite', 'plano', 'planos',
           'imagem', 'imagens', 'selo', 'edifício', 'edifícios', 'metrô',
           'fotografia', 'cidade', 'cidades', 'ícone', 'miniatura',
           'ilustração', 'mapa antigo'],
    
    'ja': ['道路', '通り', '写真', '衛星', '計画', '画像', 'スタンプ',
           '建物', '地下鉄', '都市', 'アイコン', 'イラスト', '古地図'],
    
    'zh-cn': ['道路', '街道', '照片', '卫星', '计划', '图片', '邮票',
              '建筑', '地铁', '城市', '图标', '插图', '古地图'],
    
    'ko': ['도로', '거리', '사진', '위성', '계획', '이미지', '우표',
           '건물', '지하철', '도시', '아이콘', '일러스트', '고지도'],
}

def detect_language_safe(text):
    """Bezpieczne wykrywanie języka"""
    if pd.isna(text) or str(text).strip() == '':
        return 'unknown'
    try:
        lang = detect(str(text))
        # Normalizuj kody języków
        if lang in ['zh-tw', 'zh-hk']:
            return 'zh-cn'  # Traktuj podobnie
        return lang
    except:
        return 'unknown'

def matches_criteria(row):
    """Sprawdza czy wiersz spełnia kryteria"""
    text = str(row['text']).lower() if not pd.isna(row['text']) else ''
    lang = row['detected_lang']
    
    # Dla języków bez tłumaczeń - użyj angielskich słów
    # (wiele map ma angielskie słowa nawet w innych językach)
    if lang not in positive_keywords:
        lang = 'en'
    
    pos_keywords = positive_keywords.get(lang, positive_keywords['en'])
    neg_keywords = negative_keywords.get(lang, negative_keywords['en'])
    
    # ZAWSZE sprawdź też angielskie słowa (backup)
    pos_keywords_combined = set(pos_keywords) | set(positive_keywords['en'])
    neg_keywords_combined = set(neg_keywords) | set(negative_keywords['en'])
    
    # Sprawdź pozytywne
    has_positive = any(kw.lower() in text for kw in pos_keywords_combined)
    
    # Sprawdź negatywne
    has_negative = any(kw.lower() in text for kw in neg_keywords_combined)
    
    return has_positive and not has_negative

# KROK 1: Wykryj języki
print("Wykrywanie języków...")
df_url_ok['detected_lang'] = df_url_ok['text'].apply(detect_language_safe)

print("\nRozkład języków:")
print(df_url_ok['detected_lang'].value_counts())

# KROK 2: Filtruj
print("\nFiltrowanie...")
df_filtered = df_url_ok[df_url_ok.apply(matches_criteria, axis=1)].copy()

# KROK 3: Statystyki
print(f"\n{'='*60}")
print(f"WYNIKI FILTROWANIA")
print(f"{'='*60}")
print(f"Oryginalny zbiór: {len(df_url_ok)} wierszy")
print(f"Przefiltrowany zbiór: {len(df_filtered)} wierszy")
print(f"Pozostało: {100 * len(df_filtered) / len(df_url_ok):.1f}%")

print(f"\n{'='*60}")
print("Rozkład języków w przefiltrowanym zbiorze:")
print(df_filtered['detected_lang'].value_counts())

# KROK 4: Analiza pokrycia
print(f"\n{'='*60}")
print("Pokrycie tłumaczeń:")
langs_with_translations = set(positive_keywords.keys())
for lang, count in df_filtered['detected_lang'].value_counts().head(15).items():
    coverage = "✓" if lang in langs_with_translations else "✗ (używa EN)"
    pct = 100 * count / len(df_filtered)
    print(f"{lang:10s}: {count:4d} ({pct:5.1f}%) {coverage}")

# KROK 5: Przykłady
print(f"\n{'='*60}")
print("Przykłady przefiltrowanych map:")
print(df_filtered[['text', 'detected_lang', 'url_status']].head(10))

Wykrywanie języków...

Rozkład języków:
detected_lang
en         2390
fr          280
de          257
ru          190
nl          169
it          169
tl          153
es          148
ja          143
unknown     134
ko          123
zh-cn       103
pt           95
id           89
ro           89
ca           71
pl           66
sw           64
no           59
af           57
bg           56
sl           55
da           45
vi           43
sv           41
so           38
tr           31
cs           30
et           29
ar           28
sk           23
mk           22
hr           22
fi           20
hu           19
cy           16
lt           15
el           15
sq           14
he           12
fa           11
uk           11
lv           10
bn            8
th            7
ur            2
hi            2
ta            1
Name: count, dtype: int64

Filtrowanie...

WYNIKI FILTROWANIA
Oryginalny zbiór: 5475 wierszy
Przefiltrowany zbiór: 266 wierszy
Pozostało: 4.9%

Rozkład języków w przefiltrowanym 

Pozytywne słowa (każde +1 punkt):
- choropleth
- county
- population
- density
- statistical
- demographic
- demographical
- distribution
- region
- municipality
- people
- area
- chart
- pie
- stats
- survey
- census
- bureau
- thematic

Negatywne słowa (każde -2 punkty):
- road
- street
- photo
- hiking
- satellite
- plan
- image

In [10]:
# FLAT KEYWORDS JSON
import json

data = {
    "key": ["choropleth"],
    "score": [5.0],
    "en": ["choropleth"],
    "fr": ["choroplethe"],
    "es": ["coropleta"],
    "de": ["choropleth"],
    "ko": ["등치"],
    "ru": ["хороплет"],
    "tl": ["choropleth"],
    "it": ["coropleto"],
    "nl": ["choroplet"],
    "sw": ["choropleth"],
    "ja": ["コロプレス"],
    "ca": ["coropletes"],
    "ro": ["coroplet"],
    "pl": ["choroplet"]
}

df = pd.DataFrame(data)

cols_to_iterate = [col for col in df.columns if col not in ['key', 'score']]

flat_json = {}

for _, row in df.iterrows():
    score = row['score']
    for col in cols_to_iterate:
        word = row[col]
        flat_json[word] = score

with open("flat.json", "w", encoding="utf-8") as f:
    json.dump(flat_json, f, ensure_ascii=False, indent=2)

print(json.dumps(flat_json, ensure_ascii=False, indent=2))


{
  "choropleth": 5.0,
  "choroplethe": 5.0,
  "coropleta": 5.0,
  "등치": 5.0,
  "хороплет": 5.0,
  "coropleto": 5.0,
  "choroplet": 5.0,
  "コロプレス": 5.0,
  "coropletes": 5.0,
  "coroplet": 5.0
}
